In [ ]:
import os
if os.path.exists("/kaggle/working/venv"):
    !rm -rf /kaggle/working/venv

!pip install -q virtualenv && \
 python -m virtualenv --system-site-packages /kaggle/working/venv && \
 /kaggle/working/venv/bin/pip install -q "zarr>=3.0.10" scipy "tracksdata @ git+https://github.com/royerlab/tracksdata@main" && \
 echo "===INSTALL SUCCEEDED===" || echo "===INSTALL FAILED==="

In [ ]:
!/kaggle/working/venv/bin/python -c "import torch, zarr, scipy, scipy.spatial, tracksdata as td; print('torch', torch.__version__, 'CUDA:', torch.cuda.is_available()); print('ALL IMPORTS OK')"

In [ ]:
import os
if not os.path.exists("/kaggle/working/repo"):
    !git clone -q https://github.com/royerlab/kaggle-cell-tracking-competition.git /kaggle/working/repo
!/kaggle/working/venv/bin/pip install -q -e /kaggle/working/repo && echo "===REPO INSTALL SUCCEEDED===" || echo "===REPO INSTALL FAILED==="

In [ ]:
%%writefile /kaggle/working/step4.py
import sys
sys.path.insert(0, "/kaggle/working/repo/scripts")
from dataspec import DATASET_PATH
from tracking_cellmot.io import open_dataset

sample_name = sorted(p.stem for p in DATASET_PATH.glob("*.zarr"))[0]
ds = open_dataset(DATASET_PATH / sample_name, require_tracks=True, load_image=False)
print("scale:", ds.scale)
print("image_shape:", ds.image_shape)
print("tracks nodes:", ds.tracks.num_nodes())
print("quantiles:", ds.quantiles)

In [ ]:
!/kaggle/working/venv/bin/python /kaggle/working/step4.py

In [ ]:
%%writefile /kaggle/working/step5.py
import sys
sys.path.insert(0, "/kaggle/working/repo/scripts")
from dataspec import DATASET_PATH
from tracking_cellmot.io import open_dataset

sample_name = sorted(p.stem for p in DATASET_PATH.glob("*.zarr"))[0]
ds_full = open_dataset(
    DATASET_PATH / sample_name,
    require_tracks=True, load_image=True, device="cuda",
)
print("image type:", type(ds_full.image))
print("image shape:", ds_full.image.shape)
print("image dtype:", ds_full.image.dtype)
print("image device:", ds_full.image.device)
print("image min/max:", ds_full.image.min().item(), ds_full.image.max().item())

In [ ]:
!/kaggle/working/venv/bin/python /kaggle/working/step5.py

In [ ]:
import os
DATA_DIR = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
sample_name = sorted(e for e in os.listdir(DATA_DIR) if e.endswith(".zarr"))[0][:-5]
debug_video = os.path.join(DATA_DIR, sample_name)
print("debug_video:", debug_video)

!cd /kaggle/working/repo && /kaggle/working/venv/bin/python scripts/train_unet_transformer.py --debug-video "{debug_video}" --epochs 1 --max-iters 3 --batch-size 4 --num-workers 2

In [ ]:
import os
os.makedirs("/kaggle/working/offline_wheels", exist_ok=True)
!pip wheel -q -w /kaggle/working/offline_wheels "zarr>=3.0.10" scipy "tracksdata @ git+https://github.com/royerlab/tracksdata@main" && echo "===WHEEL BUILD SUCCEEDED===" || echo "===WHEEL BUILD FAILED==="
!ls /kaggle/working/offline_wheels | wc -l
!ls /kaggle/working/offline_wheels | grep -v '\.whl$' | head

In [ ]:
!rm -rf /kaggle/working/pylibs_test
!pip install -q --no-index --find-links=/kaggle/working/offline_wheels --target=/kaggle/working/pylibs_test zarr scipy tracksdata && echo "===OFFLINE INSTALL SUCCEEDED===" || echo "===OFFLINE INSTALL FAILED==="

In [ ]:
!python -c "import sys; sys.path.insert(0, '/kaggle/working/pylibs_test'); import numpy, scipy, scipy.spatial, zarr, tracksdata; print('numpy', numpy.__version__); print('OFFLINE IMPORTS OK')"

In [ ]:
import json, os, random
from collections import defaultdict

TRAIN = "/kaggle/input/competitions/biohub-cell-tracking-during-development/train"
stems = sorted(e[:-5] for e in os.listdir(TRAIN)
               if e.endswith(".zarr") and os.path.exists(os.path.join(TRAIN, e[:-5] + ".geff")))

by_embryo = defaultdict(list)
for s in stems:
    by_embryo[s.split("_")[0]].append(s)

embryos = sorted(by_embryo)
random.Random(0).shuffle(embryos)
n_val = max(1, len(embryos) // 5)
val_emb = set(embryos[:n_val])

trn = [s for s in stems if s.split("_")[0] not in val_emb]
val = [s for s in stems if s.split("_")[0] in val_emb]

with open("/kaggle/working/dataset_splits.json", "w") as f:
    json.dump([{"split": 0, "train": trn, "test": val}], f)

print("embryos:", len(embryos), "| held out:", n_val)
print("train videos:", len(trn), "| val videos:", len(val))

In [ ]:
!cd /kaggle/working/repo && /kaggle/working/venv/bin/python scripts/train_unet_transformer.py --data-dir "/kaggle/input/competitions/biohub-cell-tracking-during-development/train" --split 0 --epochs 6 --max-iters 1000 --batch-size 8 --num-workers 4